# NLP Feature Extraction
- Part-of-Speech Tagging
- Term Frequency-Inverse Document Frequency
- Sentiment Analysis
- Pretrained Word Embeddings
- Custom Word Embbeding Model
- Speaker / Speaker history

## Import Dependencies

In [1]:
import spacy
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from typing import List, Tuple
from gensim.models import Word2Vec
import re

# Helper Functions


### 1. `load_df(filepath: str, text_column: str = "Sentence") -> pd.DataFrame`

* **Purpose**: Loads a dataset into a Pandas DataFrame from the given file path.
* **Supported Formats**: `.csv` and `.xlsx` files.
* **Functionality**:

  * Automatically detects the file extension to determine how to load the data.
  * Removes rows where the specified `text_column` (default `"Sentence"`) contains `NaN`.
  * Raises a `ValueError` if the file extension is unsupported.
  * Raises a `FileNotFoundError` if the file path is incorrect or missing.


### 2. `save_df_csv(df: pd.DataFrame, directory: str, filename: str)`

* **Purpose**: Saves a Pandas DataFrame as a CSV file to a specified directory.
* **Functionality**:

  * Automatically creates the output directory if it doesn't exist.
  * Combines the directory and filename to build a full output path.
  * Saves the DataFrame to CSV with `index=False` to exclude the index column.
  * Prints a confirmation message showing the DataFrame shape and file path.

In [2]:
def load_df(filepath: str, text_column: str = "Sentence") -> pd.DataFrame:
    """Function to load a pd.DataFrame from a given filepath"""

    # Try to read filepath as csv or excel dropping any empty sentence rows
    try:
        if filepath.endswith('.csv'):
            df = pd.read_csv(filepath).dropna(subset=[text_column]) 
        elif filepath.endswith('.xlsx'):
            df = pd.read_excel(filepath).dropna(subset=[text_column])
        else:
            raise ValueError("Unsupported file format. Please use .csv or .xlsx")
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {filepath}")
    return df[:100] # Limit to first 100 rows for testing purposes

def save_df_csv(df: pd.DataFrame, directory: str, filename: str):
    """"Function to save a Pandas dataframe as CSV"""

    # Check if output directory exists otherwise make it
    if not os.path.exists(directory):
        os.makedirs(directory)

    # Create filepath
    filepath = os.path.join(directory, filename)
    
    # Save the DataFrame to a CSV file
    df.to_csv(filepath, index=False)
    print(f"Successfully processed {df.shape} and saved the output to {filepath}")

## Create Variables

1. **Language Code Mapping**  
   - A dictionary `language_code_mapping` maps ISO language codes to their full names.  
   - This mapping ensures flexibility when working with multilingual datasets, allowing easy switching between languages.

2. **Configurable Parameters**  
   - `language`: sets the desired language for analysis (`'nl'` for Dutch in this example).  
   - `transcription_file`: specifies the input transcription dataset (CSV format).  
   - `output_file`: defines the location where extracted NLP features will be saved.

3. **Spacy Model Download Reminder**  
   - A print statement reminds the user to download the appropriate spaCy language model using:  
     ```bash
     poetry run python -m spacy download {language}_core_news_lg
     ```  
   - This ensures that the correct model for tokenization, POS tagging, and embeddings is available before feature extraction begins.


In [3]:
language_code_mapping = {
    'nl': 'Dutch',
    'en': 'English',
    'de': 'German',
    'fr': 'French',
    'es': 'Spanish',
    'it': 'Italian',
    'pt': 'Portuguese',
    'ru': 'Russian',
    'zh': 'Chinese',
    'ja': 'Japanese',
    'ko': 'Korean',
    'ar': 'Arabic',
    'hi': 'Hindi',
    'bn': 'Bengali',
    'ur': 'Urdu',
    'vi': 'Vietnamese',
    'tr': 'Turkish',
    'sv': 'Swedish',
    'da': 'Danish',
    'no': 'Norwegian',
    'fi': 'Finnish',
    'pl': 'Polish',
    'cs': 'Czech',
    'hu': 'Hungarian',
    'ro': 'Romanian',
    'el': 'Greek',
    'he': 'Hebrew',
}

language = 'nl'  # Change this to the desired language code
transcription_file = "..\\data\\group_4_url_1_transcript.csv"  # Change this to your transcription file path
output_file = '..\\data\\features\\NLP_features_output.csv'  # Change this to your desired output file path
print(f"Download model using `poetry run python -m spacy download {language}_core_news_lg` if not already installed.")

Download model using `poetry run python -m spacy download nl_core_news_lg` if not already installed.


## Function to Load Spacy Model

#### **Function: `load_spacy_model`**
- **Inputs:**
  - `language`: ISO language code specifying which spaCy model to load.  
  - `text_column` (default `'Sentence'`): The column in the dataset that contains text data.  

- **Process:**
  1. Constructs the spaCy model name dynamically: `"{language}_core_news_lg"`.  
     - Example: for Dutch (`nl`), it loads `nl_core_news_lg`.  
  2. Looks up the full language name from `language_code_mapping`.  
  3. Creates a configuration dictionary (`CONFIG`) that stores:  
     - The spaCy model name  
     - The human-readable language name   

- **Outputs:**
  - Returns a tuple:  
    1. The loaded spaCy language model.  
    2. The `CONFIG` dictionary for reference.  

- **Error Handling:**
  - If an unsupported language code is provided, a `ValueError` is raised with a clear message.


In [11]:
def load_spacy_model(
    language: str = 'nl',
    ) -> spacy.language.Language:
    """Load the spaCy model for the specified language.
    Args:
        language (str): Language code (default is 'nl' for Dutch).
    Returns:
        spacy.language.Language: Loaded spaCy language model.
    Raises:
        ValueError: If the language code is not supported.
    Authors: Floris Lokhorst, Nick Belterman
    """
    model_name = f"{language}_core_news_lg"
    try:
        language_name = language_code_mapping.get(language)
    except KeyError:
        raise ValueError(f"Unsupported language code: {language}")

    # Configuration
    CONFIG = {
        'spacy_model': model_name,
        'language': language_name,
    }

    return spacy.load(CONFIG['spacy_model']), CONFIG

## Functions to Extract POS tags

#### **Function: `extract_save_pos_tags`**
- **Inputs:**
  - `input_filepath`: Path to the dataset (CSV or Excel).  
  - `nlp_model`: The spaCy language model returned by `load_spacy_model`.  
  - `output_filepath`: Destination file for saving extracted features (default: `..\\data\\features\\NLP_features_output.csv`).  
  - `config`: Optional dictionary containing configuration settings (e.g., the text column name).  

- **Process:**
  1. Loads the dataset with `load_df`.  
  2. Determines the text column name (default: `'Sentence'`).  
  3. Ensures the output directory exists (creates it if necessary).  
  4. Loads sentences from the dataset using `load_sentences`.  
  5. Applies the spaCy pipeline to each sentence:
     - Each token in the sentence is tagged with its POS label (`token.text_POS`).  
     - The POS-tagged sentence is stored in a new DataFrame column `POS_tags`.  
  6. Adds placeholder columns `start_time` and `end_time` filled with `NaN` (likely for future alignment with temporal data).  

- **Output:**
  - A new CSV file containing the original dataset with an additional `POS_tags` column.  
  - A console message confirming the number of processed sentences and the output location.  

In [12]:
def extract_save_pos_tags(
    input_filepath: str, 
    nlp_model: spacy.language.Language, \
    output_dir: str = '..\\data\\features',
) -> None:
    """Extract POS tags from text.
    
    Authors: Floris Lokhorst, Nick Belterman
    """
    # Load df
    df  = load_df(input_filepath)

    # Get sentences
    data = df['Sentence']

    # Process each sentence and store the POS tags
    df['POS_Tags'] = data.astype(str).apply(
        lambda text: [token.pos_ for token in nlp_model(text.strip())] if text and text.strip() else []
    )
    
    save_df_csv(df, output_dir, "NLP_features_output.csv")


# Code to extract and save POS tags using spacy

In [13]:
# Load nlp model
nlp, _ = load_spacy_model()

# Extract and save POS tags
extract_save_pos_tags(
    input_filepath=transcription_file, 
    nlp_model=nlp, 
)

OSError: [E050] Can't find model 'nl_core_news_lg'. It doesn't seem to be a Python package or a valid path to a data directory.

In [ ]:
# Show POS tags
df = pd.read_csv(r"..\\data\\features\NLP_features_output.csv")
df["POS_Tags"].head()

0                    ['NOUN', 'PROPN', 'NUM', 'PUNCT']
1       ['ADV', 'ADP', 'VERB', 'DET', 'NOUN', 'PUNCT']
2    ['PRON', 'VERB', 'ADJ', 'SCONJ', 'PRON', 'PRON...
3                     ['INTJ', 'SYM', 'VERB', 'PUNCT']
4    ['PRON', 'AUX', 'DET', 'NOUN', 'ADP', 'PROPN',...
Name: POS_Tags, dtype: object

### conclusion POS tagging

POS tagging provides grammatical insights that enhance our text analysis capabilities by identifying word functions and syntactic patterns. This enables more improvised performance of downstream tasks like sentiment analysis, where adjectives and adverbs carry emotional weight. we chose spaCy for POS tagging over alternatives like NLTK or custom rule-based approaches due to its superior accuracy and processing speed. While NLTK's built-in taggers are simpler to implement, spaCy's transformer-based models provide contextual understanding crucial for our diverse text dataset.

The decision to preserve the original dataset structure while adding POS tags as a new column supports our project's modular approach. This design allows other team members to easily integrate additional NLP features without restructuring the data pipeline. Alternative approaches like separate output files would have complicated downstream processing.
We implemented list-based POS tag storage rather than string concatenation to maintain data integrity and enable easier analysis. The output shows tags like ['NOUN', 'PROPN', 'NUM', 'PUNCT'] as Python lists, which supports vectorized operations and pattern matching that our team requires for subsequent analysis phases.
Successfully processing 1050 sentences demonstrates the function's scalability and reliability. The pandas integration with lambda functions provides better performance than explicit loops while maintaining consistency with our existing data processing patterns.

# TF-IDF

# Function to load and fit TF-IDF Vectorizer on our chosen episode 

#### **Function: `load_fit_tfidf`**
- **Inputs:**
  - `sentences` (`List[str]`): A list of text sentences to process.

- **Process:**
  1. Initializes a `TfidfVectorizer` with the following settings:
     - `max_features=2000`: Limits the vocabulary to the 2,000 most important terms.  
     - `min_df=1`: Includes terms that appear in at least 1 document.  
     - `max_df=1.0`: Includes terms that appear in up to 100% of documents (all terms).  
     - `lowercase=True`: Converts all text to lowercase before tokenization.  
     - `stop_words=None`: No stopword removal is applied (all words are considered).  
     - `token_pattern=r'\b[a-záéíóúàèìòùâêîôûäëïöüç]+\b'`: Only alphabetic tokens (including accented characters) are considered.  
  2. Fits the TF-IDF vectorizer on the sentences.  
  3. Retrieves the vocabulary (list of terms included in the vectorizer).  

- **Outputs:**
  - `tfidf_matrix`: Sparse matrix where each row represents a sentence and each column represents a term, with TF-IDF weights.  
  - `vocab`: List of feature terms extracted by the vectorizer.  

- **Console Messages:**
  - Prints when TF-IDF fitting starts.  
  - Prints the total size of the vocabulary.  

In [ ]:
def load_fit_tfidf(
    sentences: List[str], 
) -> Tuple[np.ndarray, List[str]]:
    """Load and fit a TF-IDF vectorizer on the provided sentences.
    
    Args:
        sentences (List[str]): List of sentences to fit the TF-IDF vectorizer.
    
    Authors: Koen Matthijssen, Nick Belterman
    """
    # Initialize TF-IDF vectorizer class
    vectorizer = TfidfVectorizer(
        max_features=2000,         
        min_df=1,                    
        max_df=1.0,                  
        lowercase=True,
        stop_words=None,             
        token_pattern=r'\b[a-záéíóúàèìòùâêîôûäëïöüç]+\b',  
    )
    
    print("Fitting TF-IDF...")
    tfidf_matrix = vectorizer.fit_transform(sentences)
    vocab = vectorizer.get_feature_names_out()
    print(f"Total vocabulary: {len(vocab)}")
    return tfidf_matrix, vocab

# Function to add the TF-IDF matrix to our feature extraction output file

#### Function: `create_tfidf_dataframe`
- **Inputs:**
  - `output_filepath` (default: `..\\data\\features\\NLP_features_output.csv`): Path to save the resulting CSV.  
  - `text_column` (default: `'Sentence'`): Name of the text column in the original dataset.  

- **Process:**
  1. Loads the original transcription dataset and sentences.  
  2. For each sentence:
     - Converts its TF-IDF vector to an array.  
     - Extracts non-zero TF-IDF scores and maps them to words from `vocab`.  
     - Stores the TF-IDF score in a list in column `TF_IDF`.  
  3. Removes rows with missing text in the specified column.  
  4. Saves the updated DataFrame with TF-IDF features to the specified CSV file.  

- **Outputs:**
  - The CSV file now includes a `TF_IDF` column containing either readable or complete TF-IDF representations for each sentence.  
  - Console messages confirm the format used and the save location.


In [ ]:
def create_tfidf_dataframe(
    output_dir: str = '..\\data\\features',
    text_column: str = 'Sentence',
) -> pd.DataFrame:
    """Create a DataFrame from the TF-IDF matrix and vocabulary.
    
    Args:
        tfidf_matrix (np.ndarray): The TF-IDF matrix.
        vocab (List[str]): The vocabulary list.
    
    Returns:
        pd.DataFrame: DataFrame with TF-IDF features.
    
    Authors: Koen Matthijssen, Nick Belterman
    """

    # Load dataframe and sentences
    df = load_df(os.path.join(output_dir, "NLP_features_output.csv"))
    sentences = df["Sentence"]

    # Get TF-IDF matrix and vocabulary
    tfidf_matrix, vocab = load_fit_tfidf(sentences)

    # Convert to numerical lists format - only values > 0 in sentence order
    tfidf_vectors = []
    for idx, sentence in enumerate(sentences):
        vector = tfidf_matrix[idx].toarray()[0]
        sentence_words = []
        for word in sentence.lower().split():
            clean_word = re.sub(r'[^\wa-záéíóúàèìòùâêîôûäëïöüç]', '', word)
            if clean_word and clean_word in vocab:
                sentence_words.append(clean_word)
    
        sentence_order_values = []
        for word in sentence_words:
            if word in vocab:
                word_index = list(vocab).index(word)
                tfidf_score = vector[word_index]
                if tfidf_score > 0:
                    sentence_order_values.append(round(float(tfidf_score), 3))
    
        tfidf_vectors.append(sentence_order_values)

    print(f"{len(tfidf_vectors)} TF-IDF vectors created.")
    print(f"{df.shape[0]} rows in dataframe.")
    print(list(zip(sentence_words, sentence_order_values)))

    # Add to dataframe
    df['TF_IDF'] = tfidf_vectors

    # Dropna
    df.dropna(subset=[text_column], inplace=True)

    save_df_csv(df, output_dir, "NLP_features_output.csv")

# Code to extract and save TF-IDF terms using Scikit-learn

In [ ]:
create_tfidf_dataframe(
    text_column='Sentence'
)

Fitting TF-IDF...
Total vocabulary: 1353
1050 TF-IDF vectors created.
1050 rows in dataframe.
[('de', 0.251), ('auto', 0.666), ('is', 0.294), ('stuk', 0.638)]
Successfully processed (1050, 9) and saved the output to ..\data\features\NLP_features_output.csv


In [ ]:
# Show TF_IDF terms
df = pd.read_csv(r"..\\data\\features\NLP_features_output.csv")
df["TF_IDF"].head()

0                                       [0.692, 0.722]
1                  [0.409, 0.547, 0.437, 0.253, 0.528]
2    [0.342, 0.363, 0.377, 0.196, 0.342, 0.275, 0.3...
3                                       [0.679, 0.734]
4      [0.275, 0.328, 0.197, 0.47, 0.47, 0.373, 0.444]
Name: TF_IDF, dtype: object

### TF-IDF Conclusion

TF-IDF helps our video emotion classification pipeline by turning Dutch text into numbers that computers can understand. Instead of trying to analyze words directly, TF-IDF finds the most important words in each sentence and gives them scores.

This preprocessing step solves problems we found in our data analysis. It picks up unique Dutch words that show emotion while ignoring boring words like "de" and "het" that don't tell us much about feelings. The way we set it up keeps words in the right order, which matters for understanding emotions in video content.
For our pipeline, TF-IDF makes processing faster when people upload videos because it creates simple number lists instead of complicated text. It also helps deal with our unbalanced emotion data by finding special words that indicate rare emotions like disgust and fear.

The sentence-by-sentence approach works well for video processing because each part of the transcript needs its own emotion label. This catches quick emotion changes in videos and makes our pipeline work better with different types of TV shows. Using TF-IDF instead of raw text improves how well our emotion classifier performs because it gets clean, useful features rather than messy text data.

# Sentiment Analysis

- Model can be found in [Models_nlp_2526](https://edubuas-my.sharepoint.com/:f:/g/personal/225538_buas_nl/EuvOvnKbcLdPnsq8LGqDbJ0BP0DLEER7PMAwyS1bOy4JZg?e=olbDPM) and should be saved in `nlp_cia\\models\\` 
- Code used for training can be found in `nlp_cia\\src\\sentiment_analysis_train.py`

#### **Process Overview**
1. **Remove Missing Sentences**
   - Any rows in the DataFrame without a value in the `Sentence` column are dropped using `dropna(subset=['Sentence'])`.  
   - Ensures that only valid text is passed to the model.

2. **Load Model and Tokenizer**
   - `load_model_and_tokenizer(model_dir=...)` loads a pretrained model and its tokenizer from the specified directory.  
   - The tokenizer is used to convert sentences into token IDs suitable for model input.

3. **Prepare Sentences for Inference**
   - `load_inference_data(tokenizer, inference_data=df)` tokenizes the sentences in the DataFrame for model prediction.  
   - Returns tokenized `sentences` and an updated DataFrame.

4. **Generate Predictions**
   - `get_predictions(model, sentences)` runs the tokenized sentences through the model to obtain predictions.  
   - Predictions are printed to the console for verification.

5. **Save Predictions**
   - The updated DataFrame, now including model predictions, is saved to `output_file`.  

In [4]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, EvalPrediction, PreTrainedTokenizerFast
from sklearn.model_selection import train_test_split
import tensorflow as tf
from bs4 import BeautifulSoup
import re
import uuid
import json
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

c:\Users\nickb\OneDrive\Documenten\GitHub\fae2-nlpr-group-group-4-1\nlp_cia\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Functions Overview

#### `load_dataset(directory: str) -> pd.DataFrame`
Loads movie review sentences and their sentiments from a specified directory.  
- **Input**: Path to a directory containing two subfolders: `pos` (positive reviews) and `neg` (negative reviews).  
- **Output**: A `pandas.DataFrame` with two columns:  
  - `sentence`: the review text  
  - `sentiment`: `1` for positive reviews, `0` for negative reviews  

---

#### `text_cleaning(text: str) -> str`
Cleans raw review text.  
- Removes:
  - HTML tags (via BeautifulSoup)  
  - Content inside brackets (e.g., `[text]`)  
  - Special characters (keeps only letters, digits, spaces, commas, and apostrophes)  
- **Input**: Raw string of text  
- **Output**: Cleaned text string  

---

#### `compute_metrics(p: EvalPrediction) -> dict`
Computes evaluation metrics for model predictions.  
- **Input**: `EvalPrediction` object with `predictions` (logits) and `label_ids` (true labels).  
- **Process**:
  - Converts logits to predicted classes using `argmax`.  
  - Calculates **Accuracy** and **F1-score (weighted)**.  
- **Output**: Dictionary with:  
  - `accuracy`: overall accuracy score  
  - `f1`: weighted F1 score  

---

#### `encode_data(texts: list, tokenizer: PreTrainedTokenizerFast, labels: list = None, max_len: int = 128) -> dict`
Encodes text into numerical format suitable for model training.  
- **Inputs**:  
  - `texts`: list of text strings  
  - `tokenizer`: Hugging Face tokenizer  
  - `labels`: optional list of labels (`int` or `str`)  
  - `max_len`: maximum sequence length (default = 128)  
- **Process**:  
  - Tokenizes text with padding and truncation.  
  - Converts string labels (`negative`, `neutral`, `positive`) to integers if necessary.  
- **Output**: Dictionary containing PyTorch tensors:  
  - `input_ids`  
  - `attention_mask`  
  - `labels` (if provided)  

---

#### `class SentimentDataset(torch.utils.data.Dataset)`
Custom PyTorch dataset for sentiment analysis.  
- **Initialization**: Accepts an `encodings` dictionary (from `encode_data`).  
- **Methods**:  
  - `__len__`: returns number of samples.  
  - `__getitem__(idx)`: retrieves one sample at a given index, returning a dictionary with:  
    - `input_ids`  
    - `attention_mask`  
    - `labels`  

---


In [5]:
def load_dataset(directory: str) -> pd.DataFrame:
    """
    Loads movie review sentences and their sentiments from a specified directory.
    The directory is expected to contain 'pos' and 'neg' subdirectories,
    where each subdirectory contains text files with movie reviews.

    Args:
        directory: The path to the main directory containing 'pos' and 'neg' subdirectories.

    Returns:
        A pandas DataFrame with 'sentence' and 'sentiment' columns.
        'sentiment' is 1 for positive reviews and 0 for negative reviews.
    """
    data = {"sentence": [], "sentiment": []}
    for file_name in os.listdir(directory):
        print(file_name)
        # Check if the subdirectory is for positive reviews
        if file_name == 'pos':
            positive_dir = os.path.join(directory, file_name)
            # Iterate through each positive review file
            for text_file in os.listdir(positive_dir):
                text_path = os.path.join(positive_dir, text_file)
                # Read the content and append to the dataset
                with open(text_path, "r", encoding="utf-8") as f:
                    data["sentence"].append(f.read())
                    data["sentiment"].append(1) # Assign sentiment 1 for positive
        # Check if the subdirectory is for negative reviews
        elif file_name == 'neg':
            negative_dir = os.path.join(directory, file_name)
            # Iterate through each negative review file
            for text_file in os.listdir(negative_dir):
                text_path = os.path.join(negative_dir, text_file)
                # Read the content and append to the dataset
                with open(text_path, "r", encoding="utf-8") as f:
                    data["sentence"].append(f.read())
                    data["sentiment"].append(0) # Assign sentiment 0 for negative
            
    return pd.DataFrame.from_dict(data)

def text_cleaning(text: str) -> str:
    """
    Cleans a text string by removing HTML tags, bracketed content,
    and non-alphanumeric/non-whitespace characters (except commas and apostrophes).

    Args:
        text: The input string to be cleaned.

    Returns:
        The cleaned string.
    """
    # Remove HTML tags using BeautifulSoup
    soup = BeautifulSoup(text, "html.parser")
    # Remove content within brackets, e.g., "[...]"
    text = re.sub(r'\[[^]]*\]', '', soup.get_text())
    # Define a regex pattern to remove unwanted characters, keeping only
    # letters, numbers, whitespace, commas, and apostrophes
    pattern = r"[^a-zA-Z0-9\s,']"
    text = re.sub(pattern, '', text)
    return text

def compute_metrics(p: EvalPrediction) -> dict:
    """
    Computes accuracy and F1 score for model evaluation.

    Args:
        p: An EvalPrediction object containing model predictions and labels.

    Returns:
        A dictionary with 'accuracy' and 'f1' scores.
    """
    # Get the predicted class by finding the index of the max logit
    preds = np.argmax(p.predictions, axis=1)
    print(f"{preds = }")
    print(f"{p.label_ids = }")
    # Calculate accuracy and F1 score
    return {
        'accuracy': accuracy_score(p.label_ids, preds),
        'f1': f1_score(p.label_ids, preds, average='weighted'),
    }

def encode_data(texts: list, tokenizer: PreTrainedTokenizerFast, labels: list = None, max_len: int = 128) -> dict:
    """
    Encodes a list of texts into token IDs, attention masks, and optionally labels.

    Args:
        texts: A list of text strings to be encoded.
        tokenizer: The tokenizer object (e.g., from Hugging Face Transformers).
        labels: An optional list of corresponding labels for the texts. Can be integers or strings.
        max_len: The maximum sequence length for tokenization.

    Returns:
        A dictionary containing PyTorch tensors for 'input_ids', 'attention_mask',
        and 'labels' (if provided).
    """
    print(f"{texts = }")
    # Tokenize the texts with truncation and padding
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)

    # If no labels are provided, return only the encoded text data
    if labels is None:
        return {
            'input_ids': torch.tensor(encodings['input_ids']),
            'attention_mask': torch.tensor(encodings['attention_mask'])
        }
    
    # Convert string labels (if any) to integers
    if isinstance(labels[0], str):
        label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
        labels = [label_map[label] for label in labels]

    # Return a dictionary of PyTorch tensors including labels
    return {
        'input_ids': torch.tensor(encodings['input_ids']),
        'attention_mask': torch.tensor(encodings['attention_mask']),
        'labels': torch.tensor(labels)
    }

class SentimentDataset(torch.utils.data.Dataset):
    """
    A PyTorch Dataset class for handling encoded sentiment data.
    """
    def __init__(self, encodings: dict):
        """
        Initializes the dataset with encoded data.

        Args:
            encodings: A dictionary containing 'input_ids', 'attention_mask', and 'labels'.
        """
        self.encodings = encodings

    def __len__(self) -> int:
        """
        Returns the number of items in the dataset.
        """
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx: int) -> dict:
        """
        Retrieves an item (a single data sample) from the dataset at the given index.

        Args:
            idx: The index of the item to retrieve.

        Returns:
            A dictionary containing the input IDs, attention mask, and label for that item.
        """
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.encodings['labels'][idx]
        }

This script prepares and trains a **BERT model** for sentiment analysis on the **IMDb movie review dataset**. It performs several key steps: data loading, preprocessing, model setup, and training.

---

## Data Loading and Preprocessing

First, the script downloads the IMDb dataset and extracts it. It then loads the movie reviews from the `train` and `test` directories into two separate pandas **DataFrames**. The `load_dataset` function is used to read text files from `pos` and `neg` subdirectories, assigning a sentiment label of `1` for positive reviews and `0` for negative ones.

The `text_cleaning` function is then applied to both DataFrames to remove HTML tags and special characters from the text. For efficiency and testing purposes, the script limits both the training and testing datasets to the first **100 rows** before shuffling them to ensure a random distribution of positive and negative reviews. Finally, the test data is split into validation and test sets using `train_test_split`.

---

## Model Preparation and Training

A **BERT tokenizer** (`BertTokenizer`) is loaded to convert the text reviews into a format the model can understand, specifically into token IDs and attention masks. The `encode_data` function handles this tokenization and also converts the sentiment labels into PyTorch tensors. These encoded tensors are then wrapped in a custom `SentimentDataset` class, which is a standard practice for creating a PyTorch-compatible dataset.

The script then loads a pre-trained **BERT model** (`BertForSequenceClassification`) and configures it for a classification task with two labels (positive and negative). A **Hugging Face `Trainer`** is initialized with the model, training arguments (like the number of epochs and batch size), and the created datasets. The `compute_metrics` function is provided to the trainer to calculate accuracy and F1 score during evaluation.

The `trainer.train()` method starts the fine-tuning process of the BERT model on the prepared training data. After training, `trainer.evaluate()` assesses the model's performance on the test dataset.

---

## Saving the Model and Results

After training and evaluation, the script saves the fine-tuned model, its tokenizer, and the evaluation results. It creates a new directory with a unique ID to store all the relevant files, including the `eval_results.json` and `train_history.json`, allowing for future analysis and deployment.

In [6]:
# Get the current working directory
current_folder = os.getcwd()
print(current_folder)

dataset = tf.keras.utils.get_file(
    fname ="aclImdb.tar.gz", 
    origin ="http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz",
    cache_dir=  current_folder,
    extract = True)

dataset_path = os.path.dirname(dataset)
# Dataset directory
dataset_dir = os.path.join(dataset_path, 'aclImdb')

train_dir = os.path.join(dataset_dir,'train')

# Load the dataset from the train_dir
train_df = load_dataset(train_dir)

print(train_df.head())


test_dir = os.path.join(dataset_dir,'test')

# Load the dataset from the train_dir
test_df = load_dataset(test_dir)
print(test_df.head())

train_df['sentence'] = train_df['sentence'].apply(text_cleaning).tolist()
test_df['sentence'] = test_df['sentence'].apply(text_cleaning)

train_df = train_df[:100] # Limit to first 100 rows for testing purposes
test_df = test_df[:100] # Limit to first 100 rows for testing purposes

run_id = str(uuid.uuid4())[:8]
save_dir = f'models\\model_{run_id}'

# Shuffle the full dataframes to ensure labels are mixed
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Train data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")

# Training data
Reviews = train_df['Sentence'].tolist()
Target = train_df['Sentiment'].tolist()

# Test data
test_reviews = test_df['Sentence'].tolist()
test_targets = test_df['Sentiment'].tolist()
unique_types = set(type(x).__name__ for x in Reviews + Target + test_reviews + test_targets)
print(unique_types)

x_val, x_test, y_val, y_test = train_test_split(test_reviews, test_targets, test_size=0.5, stratify=test_targets)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
max_len = 128

print(f"{Reviews[0] = }\n{Target[0] = }")
train_dataset = encode_data(Reviews, tokenizer, Target)
val_dataset = encode_data(x_val, tokenizer, y_val)
test_dataset = encode_data(x_test, tokenizer, y_test)

train_ds = SentimentDataset(train_dataset)
val_ds = SentimentDataset(val_dataset)
test_ds = SentimentDataset(test_dataset)

num_labels = train_df['Sentiment'].nunique()
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels)

training_args = TrainingArguments(
    output_dir=f'{save_dir}\\results',
    num_train_epochs=1,             # For testing purposes, set to 1 epoch
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    logging_dir=f'{save_dir}\\logs',
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
eval_results = trainer.evaluate(test_ds)
print(f"Test results: {eval_results}")

run_id = str(uuid.uuid4())[:8]
model_save_dir = f'..\\models\\model_{run_id}'
os.makedirs(model_save_dir, exist_ok=True)
model.save_pretrained(model_save_dir)
tokenizer.save_pretrained(model_save_dir)
with open(os.path.join(model_save_dir, 'eval_results.json'), 'w') as f:
    json.dump(eval_results, f, indent=2)
if hasattr(trainer, 'state') and hasattr(trainer.state, 'log_history'):
    with open(os.path.join(model_save_dir, 'train_history.json'), 'w') as f:
        json.dump(trainer.state.log_history, f, indent=2)

c:\Users\nickb\OneDrive\Documenten\GitHub\fae2-nlpr-group-group-4-1\nlp_cia\src
labeledBow.feat
neg


KeyboardInterrupt: 

### Inference Functions Overview

#### `load_model_and_tokenizer(model_dir: str)`
Loads a fine-tuned BERT model and tokenizer from a given directory.  
- **Input**:  
  - `model_dir`: path to the directory containing the saved model and tokenizer.  
- **Process**:  
  - Loads `BertTokenizer` from the directory (local files only).  
  - Loads `BertForSequenceClassification` model.  
- **Output**:  
  - Tuple `(tokenizer, model)`  

---

#### `load_inference_data(tokenizer, inference_data: pd.DataFrame=None, file_path: str="data\\transcription\\csv\\transcription.csv")`
Loads and prepares sentences for inference.  
- **Inputs**:  
  - `tokenizer`: Hugging Face tokenizer  
  - `inference_data`: optional `pandas.DataFrame` with a `Sentence` column  
  - `file_path`: optional CSV path (used if `inference_data` is not provided)  
- **Process**:  
  - Loads data from DataFrame or CSV file.  
  - Cleans sentences with `text_cleaning`.  
  - Encodes sentences with `encode_data`.  
- **Outputs**:  
  - `encoded_sentences`: dictionary of tokenized input (PyTorch tensors)  
  - `df`: original DataFrame with cleaned sentences  

---

#### `get_predictions(model: BertForSequenceClassification, encodings: Dict[str, torch.Tensor]) -> List[int]`
Runs inference and produces predictions from the fine-tuned BERT model.  
- **Inputs**:  
  - `model`: fine-tuned `BertForSequenceClassification`  
  - `encodings`: dictionary with tokenized inputs (`input_ids`, `attention_mask`)  
- **Process**:  
  - Sets model to evaluation mode (`eval()`).  
  - Passes encoded inputs to the model without computing gradients.  
  - Extracts logits and converts them to predicted class labels using `argmax`.  
- **Output**:  
  - List of integer predictions (class IDs)  

---


In [ ]:
def load_model_and_tokenizer(model_dir: str):
    """
    Loads a fine-tuned BERT model and tokenizer from a specified directory.

    Args:
        model_dir (str): The directory where the model and tokenizer are saved.

    Returns:
        tuple: A tuple containing the loaded tokenizer and model.
    """
    print(f"Loading model and tokenizer from: {model_dir}")
    tokenizer = BertTokenizer.from_pretrained(model_dir, local_files_only=True)
    model = BertForSequenceClassification.from_pretrained(model_dir)
    return tokenizer, model

def load_inference_data(tokenizer, inference_data: pd.DataFrame=None, file_path: str="data\\transcription\\csv\\transcription.csv") -> list:
	"""
	Loads inference data from a CSV file.

	Args:
		file_path (str): The path to the CSV file containing inference data.
	Returns:
		list: A list of sentences for inference.
	"""
	print(os.getcwd())
	if inference_data is not None:
		df = inference_data
	else:
		df = pd.read_csv(file_path)
	sentences = df['Sentence'].apply(text_cleaning).tolist()
	encoded_sentences = encode_data(sentences, tokenizer)
	return encoded_sentences, df

In [ ]:
# Load dataframe
df = load_df(output_file)

# Load model and tokenizer
tokenizer, model = load_model_and_tokenizer(model_dir=model_save_dir)

# Load inference data
sentences, df = load_inference_data(tokenizer, inference_data=df)

# Get predictions
preds = get_predictions(model, sentences)
print(f"{preds = }")

# Store predictions in dataframe
df["Sentiment_score"] = preds

# Save dataframe
df.to_csv(output_file, index=False)

Loading model and tokenizer from: ..\\models\\Model_14e03c00
c:\Users\nickb\OneDrive\Documenten\GitHub\fae2-nlpr-group-group-4-1\nlp_cia\src
texts = ['TV GELDERLAND 2021', 'Daar boven staat een kist', 'Ik weet zeker dat ik ze alle vier gehad heb', 'O, kwaak', 'Wat moet een mens zonder Ellie Lust', 'Ellie', 'Kwaak', 'Nee, nee, nee', 'Het lijkt wel een uitgebreid gezin, de kandidatengroep van dit jaar', 'Lekker huiselijk rollen ze door het spel heen', 'Schone schijn misschien', 'Want de openheid is ingeruild voor mysteries', 'Er borrelt van alles onder het oppervlak', 'dat het verse gezinsgeluk kan verstoren', 'Bij iedere afvaller die vertrekt, valt het afscheid de groepswaarde', 'Al leek Marjolein vooral perplex te staan', 'dat juist politievrouw Ellie na een glorieuze klimpartij', 'het spel moest verlaten', 'Weg als haar hoofdverdachte', 'Of speelt de actrice dat', 'En kiest ze geheel volgens eigen script', 'gewoon weer een nieuwe nepverdachte', 'Ja, ik heb het', 'Woehoe', 'Het spel ve

### Sentiment Analysis Conclusion

Sentiment analysis can improve our emotion classification pipeline by providing an additional feature that captures an overal emotional tone of the sentence. We will achieve this by using a pre-trained Bert model for sentiment analysis fine-tuned on short text fragments (In order to deliver the code used for training we are using a different dataset). In the future we plan to use a pretrained model specifically trained on dutch text. which we will further fine-tune on our domain dataset

## Custom Word Embeddings
### We used the dutch OpenSubtitles Corpus as it matches our goal for TV emotion classification

OpenSubtitles Dutch Corpus (OPUS Collection)
Source: Movie and TV subtitle translations from OpenSubtitles.org, compiled by the OPUS project NlplPapers with Code

https://opus.nlpl.eu/OpenSubtitles/nl&en/v2024/OpenSubtitles

https://www.opensubtitles.org/en/search/subs


Scale: Part of collection spanning 60 corpora in 58 languages with 2.6 billion sentences total OpenSubtitles parallel corpora

Domain: Movies and television shows (conversational dialogue)

Language variety:  Dutch

Text type: Subtitle transcripts representing natural dialogue and speech patterns

Countries of origin: Dutch-language films and international content dubbed/subtitled in Dutch

Temporal range: Various decades of film/TV content up to recent releases

Data structure: Monolingual Dutch sentences extracted from subtitle files



## Code to extract relevant information from OpenSubtitles dataset

- **Process:**
  1. **Decompress the File**
     - Reads the `.gz` file using `gzip.open`.  
     - Writes the uncompressed content to a new file (`.gz` suffix removed).  

  2. **Read Lines and Prepare Sentences**
     - Iterates through each line of the uncompressed file.  
     - Strips whitespace and splits the line into words.  
     - Only lines with at least one word are added to the `sentences` list.  
     - The length of each line (number of words) is recorded in `line_lengths`.  
     - Stops processing once `max_sentences` is reached.

  3. **Preview and Statistics**
     - Prints the first 10 lines of the corpus for inspection.  
     - Computes and prints summary statistics:  
       - Total lines processed  
       - Number of sentences selected for training  
       - Average tokens per sentence  
       - Minimum and maximum tokens per sentence  

- **Outputs:**
  - `sentences`: List of tokenized sentences (each sentence is a list of words).  
  - `output_file`: Path to the uncompressed text file.


In [ ]:
def extract_and_prepare_corpus(filepath: str, max_sentences: int = 3_000_000):
    """Extract corpus, show statistics, and prepare for training"""
    if filepath.endswith('.gz'):
        import gzip
        # Extract the file
        filepath = filepath.replace('.gz', '')
        with gzip.open(filepath, 'rt', encoding='utf-8') as f_in:
            with open(filepath.split(".")[0], 'w', encoding='utf-8') as f_out:
                f_out.write(f_in.read())
    else:
        sentences = []
        line_lengths = []
        
        print("First 10 lines:")
        with open(filepath, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f):
                line = line.strip()
                
                # Show first 10 lines
                if line_num < 10:
                    print(f"{line_num+1}: {line}")
                
                if line:
                    words = line.split()
                    if len(words) >= 1:
                        sentences.append(words)
                        line_lengths.append(len(words))
                    
                    # Stop at max for training
                    if len(sentences) >= max_sentences:
                        break
        
        # Show statistics
        total_processed = line_num + 1
        print(f"\nDataset Statistics:")
        print(f"Total lines processed: {total_processed:,}")
        print(f"Sentences for training: {len(sentences):,}")
        print(f"Average tokens per sentence: {sum(line_lengths)/len(line_lengths):.1f}")
        print(f"Min/Max tokens: {min(line_lengths)}/{max(line_lengths)}")
        
    return sentences, filepath

In [ ]:
training_sentences, corpus_file = extract_and_prepare_corpus("..\\data\\nl.tok")

First 10 lines:
1: WERKKAMP CHANGZHOU JIANGSU , CHINA
2: VACCINATIEPROGRAMMA WERELDGEZONDHEIDSORGANISATIE
3: Wat is er ?
4: Wat is er mis ?
5: - Ga weg bij haar .
6: Blijf uit haar buurt .
7: Bel een ambulance .
8: Ze heeft een hartstilstand .
9: - Ze kan niet weg .
10: We gaan naar het ziekenhuis .

Dataset Statistics:
Total lines processed: 2,599,868
Sentences for training: 2,599,868
Average tokens per sentence: 7.1
Min/Max tokens: 1/148


#### **Function: `train_and_evaluate_model`**
- **Inputs:**
  - `sentences`: List of tokenized sentences (from `extract_and_prepare_corpus`).  
  - `params`: Dictionary of Word2Vec hyperparameters, such as `vector_size`, `window`, `min_count`, `sg`, `epochs`, and `workers`.  
  - `name`: Name of the model/configuration for identification.

- **Process:**
  1. Prints the model name and hyperparameters for clarity.  
  2. Trains a Word2Vec model using the `gensim` library:  
     - `vector_size`: Dimensionality of embeddings.  
     - `window`: Context window size.  
     - `min_count`: Ignores words with frequency below this threshold.  
     - `sg`: 1 = Skip-gram, 0 = CBOW.  
     - `epochs`: Number of training iterations.  
     - `workers`: Number of CPU threads for parallel training.  
  3. **Quick Evaluation:**  
     - Prints the vocabulary size.  
     - Checks similarity for example words (`'goed'`, `'slecht'`, `'mooi'`, `'blij'`).  
     - Prints the top 3 most similar words for each example.

- **Outputs:**
  - Returns the trained Word2Vec model instance.

#### **Testing Multiple Configurations**
- Several configurations (`configs`) are defined to test different hyperparameters:
  - `Baseline`: Standard settings.  
  - `larger_vector_size`: 500-dimensional embeddings.  
  - `smaller_window`: Reduced context window.  
  - `higher_mincount`: Ignores rarer words.  
  - `cbow_model`: Uses CBOW instead of Skip-gram.  
  - `Higher_epochs`: Trains for more epochs.  
- Each configuration is trained and stored in the `models` dictionary.


In [ ]:
def train_and_evaluate_model(sentences, params, name):
    """Train model with specific parameters and evaluate"""
    print(f"\n=== Training {name} ===")
    print(f"Parameters: {params}")
    
    model = Word2Vec(sentences=sentences, **params)
    
    # Quick evaluation
    test_words = ['goed', 'slecht', 'mooi', 'blij']
    print(f"Vocabulary size: {len(model.wv.index_to_key)}")
    
    for word in test_words:
        if word in model.wv:
            similar = model.wv.most_similar(word, topn=3)
            print(f"'{word}': {[w for w, score in similar]}")
    
    return model

# Test different configurations
configs = {
    "Baseline": {"vector_size": 200, "window": 5, "min_count": 5, "sg": 1, "epochs": 10, "workers": 8},
    "larger_vector_size": {"vector_size": 500, "window": 5, "min_count": 5, "sg": 1, "epochs": 10, "workers": 8},
    "smaller_window": {"vector_size": 200, "window": 3, "min_count": 5, "sg": 1, "epochs": 10, "workers": 8},
    "higher_mincount": {"vector_size": 200, "window": 5, "min_count": 10, "sg": 1, "epochs": 10, "workers": 8},
    "cbow_model": {"vector_size": 200, "window": 5, "min_count": 5, "sg": 0, "epochs": 10, "workers": 8},
    "Higher_epochs": {"vector_size": 200, "window": 5, "min_count": 5, "sg": 0, "epochs": 20, "workers": 8}
}

models = {}
for name, params in configs.items():
    models[name] = train_and_evaluate_model(training_sentences, params, name)


=== Training Baseline ===
Parameters: {'vector_size': 200, 'window': 5, 'min_count': 5, 'sg': 1, 'epochs': 10, 'workers': 8}
Vocabulary size: 53531
'goed': ['slecht', 'gladjes', 'uitstekend']
'slecht': ['goed', 'slechte', 'geweldig']
'mooi': ['prachtig', 'stijlvol', 'leuk']
'blij': ['Blij', 'dankbaar', 'opgelucht']

=== Training larger_vector_size ===
Parameters: {'vector_size': 500, 'window': 5, 'min_count': 5, 'sg': 1, 'epochs': 10, 'workers': 8}
Vocabulary size: 53531
'goed': ['rottig', 'geinig', 'onwerkelijk']
'slecht': ['goed', 'gehumeurd', 'geboorteberichten']
'mooi': ['prachtig', 'beeldschoon', 'stijlvol']
'blij': ['opgelucht', 'Blij', 'trouwlustig']

=== Training smaller_window ===
Parameters: {'vector_size': 200, 'window': 3, 'min_count': 5, 'sg': 1, 'epochs': 10, 'workers': 8}
Vocabulary size: 53531
'goed': ['slecht', 'amusant', 'uitstekend']
'slecht': ['goed', 'geweldig', 'Slecht']
'mooi': ['prachtig', 'fraai', 'betoverend']
'blij': ['Blij', 'opgelucht', 'dankbaar']

=== Tr

Parameter Justification:
- vector_size = 500: Balance between expressiveness and efficiency and no longer had the goed/slecht pairing
- window = 5: Captures sufficient conversational context
- min_count = 5: Filters noise, reduces vocab from 52K to 30K words
- sg = 1 (Skip-gram): Better for rare emotional vocabulary than CBOW
- epochs = 10: Sufficient convergence without overfitting

In [ ]:
print(f"Using {len(training_sentences)} sentences for training")
model = Word2Vec(
    sentences=training_sentences,
    vector_size=500,
    window=5,
    min_count=5,
    workers=8,
    epochs=10,
    sg=1,
    seed=42
)

model.save("..\\models\\dutch_custom_embeddings.model")

print(f"Training completed with {len(model.wv.index_to_key)} vocabulary words")

Using 2599868 sentences for training
Training completed with 53531 vocabulary words


In [ ]:
# Quick evaluation
test_words = ['goed', 'slecht', 'mooi', 'blij']
print(f"Vocabulary size: {len(model.wv.index_to_key)}")

for word in test_words:
    if word in model.wv:
        similar = model.wv.most_similar(word, topn=3)
        print(f"'{word}': {[w for w, score in similar]}")

Vocabulary size: 53531
'goed': ['rottig', 'gladjes', 'vlotjes']
'slecht': ['goed', 'ellendig', 'gehumeurd']
'mooi': ['prachtig', 'fraai', 'stijlvol']
'blij': ['Blij', 'opgelucht', 'trouwlustig']


### conclusion 

Domain-specific word embeddings capture semantic relationships that general pre-trained models often miss, particularly crucial for emotion classification in conversational Dutch dialogue. The OpenSubtitles corpus provided authentic conversational patterns and emotional expressions that align perfectly with our TV emotion classification objectives.
The team selected Word2Vec over alternatives like FastText or GloVe based on computational constraints and interpretability requirements. While transformer-based embeddings offer superior performance, Word2Vec's efficiency and clear semantic similarity outputs made it ideal for our iterative experimentation phase and limited computational resources.
Our systematic hyperparameter testing approach revealed important trade-offs. The larger vector size (500 dimensions) eliminated problematic word pairings like "goed/slecht" that appeared in smaller embeddings, demonstrating how dimensionality affects semantic precision. Skip-gram architecture proved superior to CBOW for capturing rare emotional vocabulary, which is essential for nuanced sentiment analysis. Successfully identifying semantic clusters like "mooi" with related aesthetic terms validates our approach. However, the observed challenge with antonyms co-occurring in similar contexts highlights the inherent limitations of distributional semantics that our team must address in downstream classification tasks.

### Observed Results:
- Successfully learned semantic clusters (e.g., "mooi" with "prachtig", "fraai")
- Identified challenge: antonyms co-occur in dialogue


## Pretrained Word Embeddings

#### **Steps and Explanation**

1. **Load Dataset**
   - Prints the current working directory for reference.  
   - Reads the CSV file `wie_is_de_mol_sentiment.csv` into a Pandas DataFrame `df`.  
   - The DataFrame is expected to contain a `sentence` column with text data.

2. **Load spaCy Model**
   - Loads the Dutch large language model: `nl_core_news_lg`.  
   - Determines the vector size (`vec_size`) of the embeddings provided by the model.

3. **Define Sentence Embedding Function**
   - `sent_embedding_spacy(text: str) -> np.ndarray`: Computes a fixed-length vector for a sentence.  
     - Converts the input text into a spaCy `Doc` object.  
     - Collects the vector of each token that has a pretrained vector.  
     - Averages token vectors to produce a single sentence-level embedding.  
     - If no tokens have vectors, returns a zero vector of length `vec_size`.

4. **Compute Sentence Embeddings**
   - Applies `sent_embedding_spacy` to each sentence in the DataFrame.  
   - Stores results in a new column `embedding`.  

5. **Stack Embeddings into Array**
   - Converts the column of individual embeddings into a single 2D NumPy array (`embeddings_array`).  
   - Prints the shape of the array: `(number of sentences, embedding dimension)`.


In [ ]:
print(os.getcwd())
df = load_df(output_file)

nlp = spacy.load("nl_core_news_lg")
vec_size = nlp.vocab.vectors_length

# Function to compute sentence embeddings
def sent_embedding_spacy(text: str) -> np.ndarray:
    doc = nlp(text)
    vecs = [t.vector for t in doc if t.has_vector]
    return np.mean(vecs, axis=0) if vecs else np.zeros(vec_size, dtype=np.float32)

print(df['Sentence'].apply(sent_embedding_spacy).head())
# Compute embeddings for each row in the 'text' column
df['Embedding'] = df['Sentence'].apply(sent_embedding_spacy)
embeddings_array = np.stack(df['Embedding'].values)
print(embeddings_array.shape)

save_df_csv(df, '..\\data\\features', "NLP_features_output.csv")

c:\Users\koenm\Documents\repositorys\year_2\fae2-nlpr-group-group-4-1\fae2-nlpr-group-group-4-1\nlp_cia\src


NameError: name 'load_df' is not defined

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Choose example sentences by index
example_indices = [0, 1, 2]  # You can change these indices
print(f"Total sentences: {len(df)}")

for idx in example_indices:
    ref_sentence = df['sentence'].iloc[idx]
    ref_embedding = df['embedding'].iloc[idx].reshape(1, -1)

    # Compute cosine similarity with all other embeddings
    sims = cosine_similarity(ref_embedding, embeddings_array)[0]
    
    # Get top 3 most similar sentence indices (excluding itself)
    top_indices = sims.argsort()[-4:][::-1]  # Get top 4, remove self below

    # Exclude itself from the results
    top_indices = [i for i in top_indices if i != idx][:3]

    print(f"\n🔹 Reference Sentence [{idx}]: {ref_sentence}")
    for i in top_indices:
        print(f"   ➤ Similar Sentence [{i}]: {df['sentence'].iloc[i]}")


### conclusion pre_trained word embedding

Here, we process text for Dutch text using spaCy's language model and create sentence embeddings by averaging word vector representations for each token. This is an appropriate text processing and embedding method for the project because it provides us with dense numerical sentence representations without adding much complexity. The choice of averaging vectors (and filling in missing ones with zeros) is one that impacts data quality since it smooths out fine details but gives a constant embedding size. Results are stored in the dataset and checked by printing the shape of the embedding, which acts to check that preprocessing steps were correctly applied. Compared to training custom word embeddings, the pretrained vectors used here are faster to integrate and already possess a general understanding of Dutch language patterns. Custom embeddings may be preferable, however, if the dataset contained very domain-specific words or senses not well represented in standard-purpose models. For the conditions in this project, the pretrained option provides a good compromise between quality and speed.

 

## Named Entity Recognition


###  `extract_save_ner_tags(...)`

```python
def extract_save_ner_tags(
    input_filepath: str, 
    nlp_model: spacy.language.Language,
    output_filepath: str = '../data/features/NLP_features_output.csv',
    config: dict = None
) -> None:
```

---

### Function Description

* **Purpose**:
  Extracts token-level NER tags from each sentence in a dataset and saves them to a new CSV file.

* **Inputs**:

  * `input_filepath` *(str)*: Path to the input `.csv` or `.xlsx` file.
  * `nlp_model` *(spaCy model)*: A loaded spaCy NLP model (e.g., `nl_core_news_lg`) for processing text.
  * `output_dir` *(str, optional)*: Destination path for saving the output CSV file. Defaults to `../data/features`.
  * `config` *(dict, optional)*: Optional dictionary to specify configuration like `text_column`. Defaults to `'Sentence'` if not provided.

* **Behavior**:

  * Loads a dataset using the `load_df()` helper.
  * Applies the spaCy model to extract NER tags for each token in the sentence.
  * Stores the tags in a new column called `NER_Tags`, where each row contains a list of entity labels (e.g., `["PER", "O", "LOC"]`).
  * Automatically creates the output directory if it does not exist.
  * Saves the augmented DataFrame to a CSV file at the specified path.

* **Output**:

  * A CSV file with an additional column `NER_Tags`.


In [ ]:
def extract_save_ner_tags(
    input_filepath: str, 
    nlp_model: spacy.language.Language, \
    output_dir: str = '..\\data\\features',
    config: dict = None
) -> None:
    """Extract NER tags from text.
    
    Authors: Nick Belterman
    """
    # Load df
    df  = load_df(input_filepath)

    # Load sentences
    data = df["Sentence"]
    entities_list = []
    for sentence in data:        
        doc = nlp_model(sentence.strip())
        sentence_entities = [token.ent_type_ if token.ent_type_ else "O" for token in doc]

        entities_list.append(sentence_entities)
    
    df["NER_Tags"] = entities_list
    
    # Save the DataFrame to a CSV file
    save_df_csv(df, output_dir, "NLP_features_output.csv")

In [ ]:
extract_save_ner_tags(
    input_filepath=output_file,
    nlp_model=nlp,
)

Successfully processed 1050 sentences and saved the output to ..\data\features\NLP_features_output.csv


In [ ]:
df = pd.read_csv(r"..\\data\\features\NLP_features_output.csv")
df["NER_Tags"].head()

### Conclusion Named Entity recognition

Named Entity recognition gives us a additional feature that has the possibility of capturing additional emotional context that can improve the accuracy of an emotional classification task based on natural language processing. A future step for our feature extraction is to incorporate all Spacy processing into a single function for maintainability and simplicity.